## Setup

First, let's set up the Python environment and import necessary libraries.

In [ ]:
import sys
from pathlib import Path

# Get the BICEP root directory (three levels up from this notebook)
bicep_root = Path.cwd().parent.parent.parent
if str(bicep_root) not in sys.path:
    sys.path.insert(0, str(bicep_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from bicep.analysis import BicepResults
from bicep.tech_adoption import TechnologyAdoption

print("Environment setup complete!")

## What Input Data Does BICEP Use?

BICEP relies on several key input datasets:

1. **Building Stock Data** - From xStock building energy models
   - Building characteristics (type, size, age, location)
   - Peak electrical loads
   - Existing electrical panel capacity

2. **Technology Adoption Forecasts** - From Scout and ReEDS
   - Heat pump adoption rates
   - Heat pump water heater adoption rates
   - Electric vehicle adoption rates
   - Solar PV adoption rates

3. **Cost Data** - For infrastructure upgrades
   - Electrical panel upgrade costs
   - Service line upgrade costs
   - Distribution transformer upgrade costs

Let's explore what technologies BICEP analyzes and how their adoption differs across scenarios.

## Available Technologies

BICEP models the following technologies and their impacts on electrical infrastructure:

In [ ]:
# Create a technologies overview
technologies = {
    'Technology': [
        'Electric Vehicles (EV)',
        'Heat Pumps (HP)',
        'Heat Pump Water Heaters (HPWH)',
        'Solar Photovoltaic (PV)'
    ],
    'End Use': [
        'Transportation',
        'Space Heating/Cooling',
        'Water Heating',
        'Electricity Generation'
    ],
    'Typical Amperage': [
        '50 A (Level 2 charger)',
        '30-60 A',
        '20-30 A',
        'Variable (reduces net load)'
    ],
    'Scenarios Available': [
        'BAU, High',
        'BAU, High',
        'BAU, High',
        'BAU, High'
    ]
}

tech_df = pd.DataFrame(technologies)
print("Technologies Analyzed by BICEP:")
print(tech_df.to_string(index=False))

## Technology Adoption Across Scenarios

Let's compare how technology adoption differs between the BAU (Business As Usual) and High electrification scenarios.

In [ ]:
# Load technology adoption data for both scenarios
bau_tech = TechnologyAdoption(scenario='bau', base_year=2020, end_year=2050)
bau_tech.calculate_adoptions()

high_tech = TechnologyAdoption(scenario='high', base_year=2020, end_year=2050)
high_tech.calculate_adoptions()

# Get residential adoption data for comparison
bau_res = bau_tech.residential.copy()
high_res = high_tech.residential.copy()

print(f"BAU residential buildings analyzed: {len(bau_res):,}")
print(f"High residential buildings analyzed: {len(high_res):,}")

### Technology Adoption Rates by Year

Let's calculate and visualize the adoption rates over time for each technology.

In [ ]:
# Calculate adoption percentages by year for residential buildings
def calculate_adoption_by_year(df, tech_columns):
    adoption_rates = {}
    for year in sorted(df['year'].unique()):
        year_data = df[df['year'] == year]
        total_buildings = len(year_data)
        adoption_rates[year] = {}
        for tech in tech_columns:
            # Only count buildings with positive adoption
            adopted = year_data[tech].sum()
            adoption_rates[year][tech] = (adopted / total_buildings * 100) if total_buildings > 0 else 0
    return pd.DataFrame(adoption_rates).T

tech_columns = ['ev_adopted', 'hp_adopted', 'hpwh_adopted', 'pv_adopted']
bau_adoption = calculate_adoption_by_year(bau_res, tech_columns)
high_adoption = calculate_adoption_by_year(high_res, tech_columns)

# Rename columns for clarity
bau_adoption.columns = ['EV', 'Heat Pump', 'HPWH', 'PV']
high_adoption.columns = ['EV', 'Heat Pump', 'HPWH', 'PV']

print("\nBAU Adoption Rates (% of buildings):")
print(bau_adoption.iloc[::5].round(1))  # Show every 5 years
print("\nHigh Adoption Rates (% of buildings):")
print(high_adoption.iloc[::5].round(1))  # Show every 5 years

In [ ]:
# Create interactive plot comparing scenarios
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Electric Vehicles', 'Heat Pumps', 'Heat Pump Water Heaters', 'Solar PV')
)

# EV adoption
fig.add_trace(
    go.Scatter(x=bau_adoption.index, y=bau_adoption['EV'], name='BAU', line=dict(dash='dash')),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=high_adoption.index, y=high_adoption['EV'], name='High', line=dict(dash='solid')),
    row=1, col=1
)

# Heat Pump adoption
fig.add_trace(
    go.Scatter(x=bau_adoption.index, y=bau_adoption['Heat Pump'], name='BAU', showlegend=False, line=dict(dash='dash')),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=high_adoption.index, y=high_adoption['Heat Pump'], name='High', showlegend=False, line=dict(dash='solid')),
    row=1, col=2
)

# HPWH adoption
fig.add_trace(
    go.Scatter(x=bau_adoption.index, y=bau_adoption['HPWH'], name='BAU', showlegend=False, line=dict(dash='dash')),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=high_adoption.index, y=high_adoption['HPWH'], name='High', showlegend=False, line=dict(dash='solid')),
    row=2, col=1
)

# PV adoption
fig.add_trace(
    go.Scatter(x=bau_adoption.index, y=bau_adoption['PV'], name='BAU', showlegend=False, line=dict(dash='dash')),
    row=2, col=2
)
fig.add_trace(
    go.Scatter(x=high_adoption.index, y=high_adoption['PV'], name='High', showlegend=False, line=dict(dash='solid')),
    row=2, col=2
)

# Update layout
fig.update_xaxes(title_text="Year", row=2, col=1)
fig.update_xaxes(title_text="Year", row=2, col=2)
fig.update_yaxes(title_text="Adoption Rate (%)", row=1, col=1)
fig.update_yaxes(title_text="Adoption Rate (%)", row=1, col=2)
fig.update_yaxes(title_text="Adoption Rate (%)", row=2, col=1)
fig.update_yaxes(title_text="Adoption Rate (%)", row=2, col=2)

fig.update_layout(
    title_text="Technology Adoption Rates: BAU vs High Scenario (Residential)",
    height=700,
    showlegend=True
)
fig.show()

## Key Differences Between Scenarios

Let's quantify the differences in technology adoption between scenarios.

In [ ]:
# Compare 2050 adoption rates
year_2050_bau = bau_adoption.loc[2050]
year_2050_high = high_adoption.loc[2050]
difference = year_2050_high - year_2050_bau

comparison_df = pd.DataFrame({
    'Technology': year_2050_bau.index,
    'BAU 2050': year_2050_bau.values,
    'High 2050': year_2050_high.values,
    'Difference': difference.values
})

print("\n2050 Adoption Rate Comparison (% of buildings):")
print(comparison_df.to_string(index=False))
print(f"\nKey Finding: The 'High' scenario shows significantly higher adoption rates,")
print(f"especially for heat pumps and heat pump water heaters.")

## Using Custom Technology Forecasts

BICEP is designed to work with forecasts from Scout and ReEDS, but you can also integrate custom forecasts. Here's how you would approach this:

### Option 1: Modify Adoption Data Before BICEP Analysis

If you have custom adoption forecasts, you can replace the adoption columns in the residential/commercial DataFrames before analysis.

In [ ]:
# Example: Working with custom adoption data
# This shows how you could inject custom forecasts

from bicep.tech_adoption import TechnologyAdoption

# Create a TechnologyAdoption instance
tech = TechnologyAdoption(scenario='bau')
tech.calculate_adoptions()

# Show current adoption columns
adoption_cols = [col for col in tech.residential.columns if 'adopted' in col]
print("Current adoption columns:")
print(adoption_cols)

# You can modify these columns with custom data
# Example: Set a custom EV adoption rate
# tech.residential.loc[tech.residential['year'] == 2030, 'ev_adopted'] = custom_ev_rate

print("\nCustom adoption data can be injected by:")
print("1. Loading your forecasts into a DataFrame")
print("2. Joining them with the building data by state/year")
print("3. Updating the adoption columns in tech.residential or tech.commercial")
print("4. Running the cost analysis on the modified data")

### Option 2: Provide Your Own Forecast File

If you have adoption forecasts in a CSV file, you can merge them with BICEP's building data:

In [ ]:
# Example structure for custom adoption data
custom_forecast_structure = pd.DataFrame({
    'state': ['CA', 'CA', 'TX', 'TX'],
    'year': [2030, 2040, 2030, 2040],
    'ev_adoption_rate': [0.25, 0.60, 0.20, 0.50],  # Fraction of buildings
    'hp_adoption_rate': [0.30, 0.70, 0.25, 0.65],
    'hpwh_adoption_rate': [0.20, 0.50, 0.15, 0.45],
    'pv_adoption_rate': [0.15, 0.40, 0.10, 0.35]
})

print("Example format for custom adoption forecasts:")
print(custom_forecast_structure.to_string(index=False))

print("\n" + "="*60)
print("To use custom forecasts:")
print("="*60)
print("1. Prepare data in the format above (state, year, adoption rates)")
print("2. Load your CSV: custom_data = pd.read_csv('your_forecasts.csv')")
print("3. Merge with BICEP building data by state and year")
print("4. Use values in custom_data columns to override adoption columns")
print("5. Run BicepResults on the modified data")

## Summary

### Key Takeaways:

1. **BICEP analyzes 4 technologies**: EVs, Heat Pumps, HPWHs, and Solar PV
2. **Two scenarios are available**: BAU (lower adoption) and High (higher adoption)
3. **Technology adoption varies significantly**: The High scenario shows 2-3x higher adoption rates for heating technologies
4. **Custom forecasts can be integrated**: BICEP can work with your own adoption forecasts by merging data by state and year

### Next Steps:

- Explore the [Scenario Comparison](scenario-comparison.ipynb) notebook to see how these adoption differences affect infrastructure costs
- Check the [API Reference](../api-reference.md) for detailed method documentation
- Review the [Custom Distributions](custom-distributions.ipynb) notebook to learn about cost distributions